#### Decision API 2차 배치 검수

`ner_human_review.review_method = 'LLM검수'` 대상만 PostgreSQL에서 스트리밍 추출하고, 언론사별로 태그 오류와 태깅 병합 필요 여부를 판단한다.

- API 실행은 맨 마지막 `AUTO_RUN` 셀 하나가 토큰 게이트와 10건 테스트를 거쳐 자동 진행한다.
- 안전 기준 초과 또는 10건 테스트 실패 시 전체 실행은 자동 중단한다.
- 완료된 식별자는 언론사별 `results.jsonl`에서 읽어 재호출하지 않는다.

In [9]:
import os
import sys
from datetime import datetime
from itertools import islice
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(r'C:\work\ner')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_postgre_connect
from src.ner_review.decision.decision import DECISION_MODEL
from src.ner_review.decision.batch import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_WORKERS,
    DEFAULT_MAX_RETRIES,
    DEFAULT_REQUEST_TIMEOUT_SECONDS,
    build_decision_input,
    estimate_usage,
    run_press_list,
    stream_review_targets,
)

DATA_VERSION = "20261117_192958_강원일보" # os.getenv('DATA_VERSION') or None
MODEL_OVERRIDE = None  # Decisions API 공식 지원 모델 외 값은 사용하지 않음
MODEL = MODEL_OVERRIDE or DECISION_MODEL
if MODEL != 'gpt-6-luna':
    raise ValueError(f'Decisions API 지원 모델은 gpt-6-luna입니다: {MODEL}')
RUN_API = False  # 구형 실행 셀 비활성화. 실제 실행은 맨 마지막 AUTO_RUN 셀에서 수행
PRESS_FILTER = None  # 예: ['강원일보']
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'decision_2' / f'run_{datetime.now():%Y%m%d_%H%M%S}'

# API 제한과 비용에 맞춰 조정하는 설정
BATCH_SIZE = DEFAULT_BATCH_SIZE
MAX_WORKERS = DEFAULT_MAX_WORKERS
MAX_RETRIES = DEFAULT_MAX_RETRIES
BACKOFF_SECONDS = 2.0
MAX_BACKOFF_SECONDS = 60.0
REQUEST_TIMEOUT_SECONDS = DEFAULT_REQUEST_TIMEOUT_SECONDS
MAX_INPUT_CHARS = 120_000
SAMPLE_SIZE = 100
CHARS_PER_TOKEN = 4.0
ESTIMATED_OUTPUT_TOKENS_PER_REQUEST = 300
INPUT_PRICE_PER_1M = None   # 모델 가격표 확인 후 숫자로 입력
OUTPUT_PRICE_PER_1M = None  # 모델 가격표 확인 후 숫자로 입력

print('output_root:', OUTPUT_ROOT)
print('model:', MODEL, '| run_api:', RUN_API)

output_root: C:\work\ner\outputs\decision_2\run_20261119_233320
model: gpt-6-luna | run_api: False


##### 1. 대상 언론사와 소규모 입력 확인

In [10]:
def list_press_names(data_version=None):
    where = "WHERE review_method = 'LLM검수'"
    params = []
    if data_version:
        where += ' AND data_version = %s'
        params.append(data_version)
    with get_postgre_connect() as connection:
        rows = connection.execute(
            f'SELECT DISTINCT press FROM ner_human_review {where} ORDER BY press',
            params,
        ).fetchall()
    return [row[0] for row in rows]

press_list = list_press_names(DATA_VERSION)
if PRESS_FILTER:
    press_list = [press for press in press_list if press in PRESS_FILTER]
print('presses:', press_list)

preview_rows = []
for press in press_list[:3]:
    target = next(stream_review_targets(data_version=DATA_VERSION, press=press), None)
    if target:
        preview_rows.append({
            'press': press,
            'data_version': target['data_version'],
            'doc_id': target['doc_id'],
            'sentence_id': target['sentence_id'],
            'original_form': target['original_form'],
            'entity_count': len(target['entities']),
            'active_rule_count': len(target['sentence_rule_evidence']),
        })
display(pd.DataFrame(preview_rows))

presses: ['강원일보']


,press,data_version,doc_id,sentence_id,original_form,entity_count,active_rule_count
0,강원일보,20261117_192958_강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,<2022년: DT> <20대 대선: EV>의 <강원: LC>지역 <1일차: DT>...,1,1


##### 2. 언론사별 요청 수·토큰·비용 추정

전체 입력은 메모리에 모으지 않고, 언론사별 generator를 한 번 순회한다. 토큰은 문자 수 기반 보수적 추정이며 실제 사용량은 결과 JSONL의 `api.usage`를 확인한다.

In [11]:
usage_rows = []
for press in press_list:
    target_iter = stream_review_targets(data_version=DATA_VERSION, press=press)
    sample_targets = list(islice(target_iter, SAMPLE_SIZE))
    usage = estimate_usage(sample_targets, sample_size=SAMPLE_SIZE, chars_per_token=CHARS_PER_TOKEN)
    remaining_count = sum(1 for _ in target_iter)
    estimated_request_count = len(sample_targets) + remaining_count
    estimated_input_tokens = usage['estimated_input_tokens']
    estimated_input_tokens_total = estimated_request_count * estimated_input_tokens
    estimated_output_tokens_total = estimated_request_count * ESTIMATED_OUTPUT_TOKENS_PER_REQUEST
    estimated_cost = None
    if INPUT_PRICE_PER_1M is not None and OUTPUT_PRICE_PER_1M is not None:
        estimated_cost = (estimated_input_tokens_total / 1_000_000 * INPUT_PRICE_PER_1M) + (estimated_output_tokens_total / 1_000_000 * OUTPUT_PRICE_PER_1M)
    usage_rows.append({
        'press': press,
        'sample_size': usage['sample_size'],
        'estimated_request_count': estimated_request_count,
        'avg_input_chars': usage['avg_input_chars'],
        'max_input_chars': usage.get('max_input_chars', 0),
        'estimated_input_tokens_per_request': estimated_input_tokens,
        'estimated_input_tokens_total': estimated_input_tokens_total,
        'estimated_output_tokens_total': estimated_output_tokens_total,
        'estimated_cost': estimated_cost,
    })
usage_frame = pd.DataFrame(usage_rows)
display(usage_frame)
print('주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.')

,press,sample_size,estimated_request_count,avg_input_chars,max_input_chars,estimated_input_tokens_per_request,estimated_input_tokens_total,estimated_output_tokens_total,estimated_cost
0,강원일보,100,2177,504.8,765,126,274302,653100,None


주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.


##### 3. 언론사별 순차 배치 실행

`MAX_WORKERS`만 한 언론사 내부에서 병렬 처리하고, 언론사 자체는 순차 처리한다. 배치가 끝날 때마다 `results.jsonl`, `failures.jsonl`, `summary.json`, `execution.log.jsonl`을 저장한다.

In [ ]:
if RUN_API:
    summaries = run_press_list(
        press_list,
        OUTPUT_ROOT,
        data_version=DATA_VERSION,
        model=MODEL,
        batch_size=BATCH_SIZE,
        max_workers=MAX_WORKERS,
        max_retries=MAX_RETRIES,
        backoff_seconds=BACKOFF_SECONDS,
        max_backoff_seconds=MAX_BACKOFF_SECONDS,
        request_timeout_seconds=REQUEST_TIMEOUT_SECONDS,
        max_input_chars=MAX_INPUT_CHARS,
        progress=lambda event: print(event),
    )
    display(pd.DataFrame(summaries))
else:
    print('RUN_API=False: 실제 Decision API 호출을 실행하지 않았습니다.')

##### 4. 실행 현황과 결과 확인

In [ ]:
def load_summaries(output_root):
    paths = sorted(Path(output_root).glob('press/*/summary.json'))
    rows = []
    for path in paths:
        rows.append(__import__('json').loads(path.read_text(encoding='utf-8')).copy())
        rows[-1]['summary_path'] = str(path)
    return pd.DataFrame(rows)

summary_frame = load_summaries(OUTPUT_ROOT)
display(summary_frame)
print('결과 루트:', OUTPUT_ROOT)

In [ ]:
# 재개 실행 예시: 같은 OUTPUT_ROOT를 사용하면 completed 식별자를 건너뛴다.
# 실패 건은 failures.jsonl을 확인한 뒤 필요하면 같은 press만 다시 실행한다.
# run_press('강원일보', OUTPUT_ROOT, data_version=DATA_VERSION, model=MODEL,
#           batch_size=BATCH_SIZE, max_workers=MAX_WORKERS, max_retries=MAX_RETRIES)

##### 실제 실행 순서(맨 마지막 AUTO_RUN 셀 하나만 실행)

1. 위 설정 셀에서 `DATA_VERSION`, `PRESS_FILTER`, `BATCH_SIZE`, `MAX_WORKERS`를 확인한다. 모델은 `gpt-6-luna`로 고정된다.
2. AUTO_RUN 셀 하나가 토큰 추정치를 확인한다. 요청당 입력 토큰·입력 문자·전체 예상 토큰이 안전 기준을 넘으면 API를 호출하지 않고 중단한다.
3. 안전 기준을 통과하면 첫 언론사 10건을 자동 호출하고 결과 파일을 자동 확인한다.
4. 10건 테스트에서 실패가 없을 때만 전체 언론사 순차 실행으로 자동 진행한다.
5. 중단 후 동일 `OUTPUT_ROOT`로 재실행하면 완료 식별자는 건너뛰고 실패·미처리 건만 다시 대상이 된다.

In [12]:
# 순서 2: 토큰/요청 수 확인(Decision API 미호출)
display(usage_frame[['press', 'estimated_request_count', 'estimated_input_tokens_total', 'estimated_output_tokens_total', 'estimated_cost']])

,press,estimated_request_count,estimated_input_tokens_total,estimated_output_tokens_total,estimated_cost
0,강원일보,2177,274302,653100,None


In [13]:
# 순서 3은 맨 마지막 자동 실행 셀에서 안전 게이트 후 수행한다.
print('10건 테스트는 맨 마지막 AUTO_RUN 셀에서 자동 수행됩니다.')

10건 테스트는 맨 마지막 AUTO_RUN 셀에서 자동 수행됩니다.


In [14]:
# 순서 4는 10건 테스트 직후 맨 마지막 자동 실행 셀에서 수행한다.
print('테스트 결과 확인도 맨 마지막 AUTO_RUN 셀에서 자동 수행됩니다.')

테스트 결과 확인도 맨 마지막 AUTO_RUN 셀에서 자동 수행됩니다.


In [15]:
# 순서 5도 맨 마지막 AUTO_RUN 셀에서 테스트 성공 후 자동 수행한다.
print('전체 실행은 테스트 성공 시 맨 마지막 AUTO_RUN 셀에서 자동 시작됩니다.')

전체 실행은 테스트 성공 시 맨 마지막 AUTO_RUN 셀에서 자동 시작됩니다.


In [ ]:
# AUTO_RUN: 토큰 확인 -> 안전 게이트 -> 10건 테스트 -> 결과 확인 -> 전체 실행
import importlib
import src.ner_review.decision.batch as batch_module
importlib.reload(batch_module)
from src.ner_review.decision.batch import count_completed_results, run_press_with_remediation

MAX_SAFE_INPUT_CHARS_PER_REQUEST = MAX_INPUT_CHARS
MAX_SAFE_INPUT_TOKENS_PER_REQUEST = 12000
MAX_SAFE_TOTAL_TOKENS = 2000000
TEST_LIMIT = 10

if not press_list:
    raise RuntimeError('LLM검수 대상 언론사가 없습니다.')

unsafe_rows = usage_frame[
    (usage_frame['max_input_chars'] > MAX_SAFE_INPUT_CHARS_PER_REQUEST)
    | (usage_frame['estimated_input_tokens_per_request'] > MAX_SAFE_INPUT_TOKENS_PER_REQUEST)
].copy()
total_input_tokens = int(usage_frame['estimated_input_tokens_total'].sum())
total_output_tokens = int(usage_frame['estimated_output_tokens_total'].sum())
total_estimated_tokens = total_input_tokens + total_output_tokens
print(f'예상 전체 입력 토큰: {total_input_tokens:,}')
print(f'예상 전체 출력 토큰: {total_output_tokens:,}')
print(f'안전 기준: 요청당 입력 {MAX_SAFE_INPUT_TOKENS_PER_REQUEST:,} tokens, 전체 {MAX_SAFE_TOTAL_TOKENS:,} tokens')

if not unsafe_rows.empty:
    display(unsafe_rows[['press', 'max_input_chars', 'estimated_input_tokens_per_request']])
    raise RuntimeError('입력 크기 또는 요청당 토큰이 안전 기준을 초과하여 API 실행을 중단했습니다.')
if total_estimated_tokens > MAX_SAFE_TOTAL_TOKENS:
    raise RuntimeError('전체 예상 입력+출력 토큰이 안전 기준을 초과하여 API 실행을 중단했습니다.')

test_press = press_list[0]
expected_test_count = min(TEST_LIMIT, int(usage_frame.loc[usage_frame['press'] == test_press, 'estimated_request_count'].iloc[0]))
test_output_root = PROJECT_ROOT / 'outputs' / 'decision_2_test' / f'run_{datetime.now():%Y%m%d_%H%M%S}'
test_summary = run_press_with_remediation(
    test_press, test_output_root, data_version=DATA_VERSION, model=MODEL, limit=TEST_LIMIT,
    batch_size=min(BATCH_SIZE, TEST_LIMIT), max_workers=MAX_WORKERS, max_retries=MAX_RETRIES,
    backoff_seconds=BACKOFF_SECONDS, max_backoff_seconds=MAX_BACKOFF_SECONDS,
    request_timeout_seconds=REQUEST_TIMEOUT_SECONDS, max_input_chars=MAX_INPUT_CHARS,
    progress=lambda event: print(event),
)
test_dir = test_output_root / 'press' / test_press
test_failed = int(test_summary.get('failed', 0))
test_completed = count_completed_results(test_output_root, test_press)
print(f'10건 테스트 완료: completed={test_completed}/{expected_test_count}, failed={test_failed}, remediation={test_summary.get("remediation")}')
if (test_dir / 'results.jsonl').exists():
    result_frame = pd.read_json(test_dir / 'results.jsonl', lines=True)
    result_columns = ['press', 'doc_id', 'sentence_id', 'tagging_error', 'tagging_merge', 'status']
    display(result_frame[[column for column in result_columns if column in result_frame.columns]])
if (test_dir / 'failures.jsonl').exists():
    display(pd.read_json(test_dir / 'failures.jsonl', lines=True))
if test_failed or test_completed < expected_test_count:
    raise RuntimeError('자동 복구 후에도 10건 테스트가 완료되지 않아 전체 실행을 중단했습니다. failures.jsonl을 확인하세요.')

full_summaries = run_press_list(
    press_list, OUTPUT_ROOT, data_version=DATA_VERSION, model=MODEL,
    batch_size=BATCH_SIZE, max_workers=MAX_WORKERS, max_retries=MAX_RETRIES,
    backoff_seconds=BACKOFF_SECONDS, max_backoff_seconds=MAX_BACKOFF_SECONDS,
    request_timeout_seconds=REQUEST_TIMEOUT_SECONDS, max_input_chars=MAX_INPUT_CHARS,
    progress=lambda event: print(event),
)
display(pd.DataFrame(full_summaries))

예상 전체 입력 토큰: 274,302
예상 전체 출력 토큰: 653,100
안전 기준: 요청당 입력 12,000 tokens, 전체 2,000,000 tokens
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10014', 'sentence_id': 'NLRW2600000001.10014.6.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10014', 'sentence_id': 'NLRW2600000001.10014.1.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10008', 'sentence_id': 'NLRW2600000001.10008.5.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10006', 'sentence_id': 'NLRW2600000001.10006.6.2', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.1002', 'sentence_id': 'NLRW2600000001.1002.4.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10024', 'sentence_id': 'NLRW2600000001.10024.6.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10031', 'sentence_id': 'NLRW2600000001.10031.2.1', 'a

,press,doc_id,sentence_id,tagging_error,tagging_merge,status
0,강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,"{'is_error': True, 'error_type': 'other_taggin...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
1,강원일보,NLRW2600000001.10008,NLRW2600000001.10008.5.1,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
2,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.1.1,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
3,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.6.1,"{'is_error': True, 'error_type': 'other_taggin...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
4,강원일보,NLRW2600000001.1002,NLRW2600000001.1002.4.1,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
5,강원일보,NLRW2600000001.10024,NLRW2600000001.10024.6.1,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
6,강원일보,NLRW2600000001.10031,NLRW2600000001.10031.2.1,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
7,강원일보,NLRW2600000001.10031,NLRW2600000001.10031.7.2,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
8,강원일보,NLRW2600000001.1004,NLRW2600000001.1004.7.1,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed
9,강원일보,NLRW2600000001.1004,NLRW2600000001.1004.9.1,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",completed


""


{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10014', 'sentence_id': 'NLRW2600000001.10014.1.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10008', 'sentence_id': 'NLRW2600000001.10008.5.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10006', 'sentence_id': 'NLRW2600000001.10006.6.2', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10014', 'sentence_id': 'NLRW2600000001.10014.6.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10031', 'sentence_id': 'NLRW2600000001.10031.2.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10024', 'sentence_id': 'NLRW2600000001.10024.6.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.1002', 'sentence_id': 'NLRW2600000001.1002.4.1', 'attempts': 1}
{'event': 'completed', 'press': '강원일보', 'doc_id': 'NLRW2600000001.10031', 'sen